# Lezione 7 — Vettori, matrici e tensori: il linguaggio delle forme

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 6 (pensare per array invece che per cicli, e il parametro `axis` nelle riduzioni).

**Cosa saprai fare alla fine:** leggere e prevedere le **forme** (shape) dei
dati in qualsiasi pipeline — la competenza che distingue chi capisce un
modello da chi copia codice. La metà degli errori in TensorFlow sono errori
di forma: dopo questa lezione sapranno di cosa parlano.

## Parte 1 — Teoria: tre nomi per la stessa idea

- **Vettore** — una lista di numeri, forma `(n,)`. Nel nostro progetto: le
  7 feature di *una* memoria sono un vettore di forma `(7,)`.
- **Matrice** — una tabella, forma `(righe, colonne)`. Le 213 memorie del
  train sono una matrice `(213, 7)`: **prima dimensione = esempi, seconda =
  feature**. Questa convenzione è universale nel machine learning.
- **Tensore** — il termine generale: array a n dimensioni. Un vettore è un
  tensore 1D, una matrice un tensore 2D. Un'immagine a colori è un tensore
  3D `(altezza, larghezza, 3)`; un batch di 32 immagini è 4D
  `(32, altezza, larghezza, 3)`. Quando TensorFlow dice "tensor" intende
  questo — niente di più esotico.

La regola pratica: **davanti a qualsiasi dato, la prima domanda è "che forma
ha?"**. La risposta dice cosa puoi farci.

## Parte 2 — Teoria: il prodotto scalare è una somma pesata

L'operazione fondamentale di tutto il machine learning è banale: il
**prodotto scalare** (dot product) tra due vettori — moltiplica elemento per
elemento e somma:

`punteggio = x · w = x[0]*w[0] + x[1]*w[1] + ... + x[6]*w[6]`

Leggilo come una **somma pesata**: ogni feature `x[i]` contribuisce al
punteggio secondo il suo peso `w[i]`. Un peso grande e positivo = "questa
feature spinge il punteggio su"; negativo = "lo spinge giù"; vicino a zero =
"irrilevante". *Un neurone di una rete neurale è esattamente questo*: una
somma pesata più una soglia.

E la **moltiplicazione matrice-vettore** `X @ w`? È il prodotto scalare
applicato a ogni riga: 213 somme pesate in un colpo solo — un punteggio per
ogni memoria. E' la stessa idea di "vettorizzare invece di iterare" della Lezione 6, applicata all'operazione piu' importante del corso.

Il **broadcasting** completa il quadro: quando le forme non coincidono ma
sono compatibili, NumPy "estende" la più piccola. `X - media` con `X` di
forma `(213, 7)` e `media` di forma `(7,)` sottrae la media giusta da ogni
riga - e' lo stesso meccanismo che hai gia' usato nella Lezione 5 per standardizzare le feature (`(X - media) / dev`), senza che il notebook te lo dicesse esplicitamente allora.

In [ ]:
import numpy as np

x = np.array([2.0, -1.0, 0.5])      # una memoria con 3 feature
w = np.array([1.0, 0.0, 4.0])       # i pesi: la seconda feature non conta

print('a mano  :', x[0] * w[0] + x[1] * w[1] + x[2] * w[2])
print('con  @  :', x @ w)

X = np.array([[2.0, -1.0, 0.5],
              [0.0,  3.0, 1.0],
              [1.0,  1.0, 1.0]])    # 3 memorie x 3 feature
print('X @ w   :', X @ w, '  <- un punteggio per riga, in un colpo solo')

Vediamo il broadcasting con un esempio esplicito, incluso il caso in cui le forme **non** sono compatibili - e' l'errore piu' comune quando si combinano array di dimensioni diverse:

In [ ]:
X = np.array([[18.0, 60.0],
             [20.0, 55.0],
             [16.0, 70.0]])      # 3 righe x 2 colonne
media = np.array([18.0, 61.7])    # 1 valore per colonna, forma (2,)

print('X - media (compatibile, la media si applica a ogni riga):')
print(X - media)

vettore_sbagliato = np.array([1.0, 2.0, 3.0])  # 3 valori, ma X ha 2 colonne
try:
    X - vettore_sbagliato
except ValueError as errore:
    print('\nX - vettore_sbagliato fallisce:', errore)

`X` ha forma `(3, 2)` e `media` ha forma `(2,)`: NumPy allinea le dimensioni **da destra** e vede che l'ultima di entrambe e' `2` - compatibile, quindi ripete `media` su ognuna delle 3 righe. `vettore_sbagliato` ha invece forma `(3,)`: l'ultima dimensione e' `3`, che non combacia con il `2` di `X`, e NumPy si rifiuta di indovinare cosa intendevi. La regola generale: due dimensioni sono compatibili se sono uguali, oppure se una delle due e' `1`; altrimenti e' un errore, non un'approssimazione silenziosa.

## Parte 3 — Teoria: leggere le forme di una moltiplicazione

La regola delle forme per `A @ B`:

```text
(n, k) @ (k, m) -> (n, m)     le dimensioni interne devono combaciare
```

Esempi che incontrerai continuamente:

- `(213, 7) @ (7,)   -> (213,)`   — un punteggio per esempio;
- `(213, 7) @ (7, 4) -> (213, 4)` — **quattro** punteggi per esempio: uno
  per classe. Questa è la forma di un classificatore lineare a 4 classi, ed
  è esattamente quello che costruiremo nella Lezione 9;
- `(32, 784) @ (784, 128) -> (32, 128)` — uno strato denso di una rete
  neurale su un batch di 32 esempi. Già ora sai leggerlo.

## Parte 4 — Esercizio guidato

Prima di eseguire, **scrivi su carta** la forma del risultato di ognuna:

```python
A = np.ones((4, 3))
B = np.ones((3, 5))
v = np.ones(3)
1) A @ B        2) A @ v        3) B @ A        4) A - v
```

Poi verificale nella cella qui sotto, e infine calcola con **un solo `@`**
il punteggio `2*temperatura + 0.5*umidita'` per queste tre letture:

In [ ]:
letture = np.array([[18.0, 60.0],
                    [31.0, 40.0],
                    [15.0, 75.0]])   # 3 letture x (temperatura, umidita')

# Scrivi qui:
# 1) verifica le 4 forme previste (attenzione: una delle 4 e' un errore!)
# 2) pesi = ... ; punteggi = letture @ pesi

letture.shape

### Soluzione spiegata

- `A @ B`: (4,3) @ (3,5) → **(4,5)**; `A @ v`: (4,3) @ (3,) → **(4,)**;
- `B @ A`: (3,5) @ (4,3) → **errore**: 5 ≠ 4, le dimensioni interne non
  combaciano — l'ordine nella moltiplicazione di matrici conta;
- `A - v`: broadcasting (4,3) con (3,) → **(4,3)**, v sottratto a ogni riga;
- il punteggio è una somma pesata: basta mettere i pesi in un vettore.

In [ ]:
A, B, v = np.ones((4, 3)), np.ones((3, 5)), np.ones(3)
assert (A @ B).shape == (4, 5)
assert (A @ v).shape == (4,)
assert (A - v).shape == (4, 3)
try:
    B @ A
except ValueError as errore:
    print('B @ A fallisce come previsto:', errore)

pesi = np.array([2.0, 0.5])
punteggi = letture @ pesi
print('punteggi:', punteggi)

## Parte 5 — Il progetto: Memory AI Lab, passo 7

Oggi il progetto fa il gesto fondamentale: **produce punteggi di classe con
una moltiplicazione di matrici**. I pesi per ora sono casuali — quindi il
"classificatore" tirerà a indovinare — ma la *struttura* è quella vera:

```text
(213 memorie, 7 feature) @ (7 feature, 4 classi) -> (213, 4 punteggi)
```

La classe predetta è quella col punteggio più alto (`argmax`). Con pesi
casuali l'accuratezza sarà vicina al caso: quel numero è la **baseline** che
le Lezioni 8 e 9 dovranno battere *imparando* i pesi.

In [ ]:
import pandas as pd
from pathlib import Path

train = pd.read_csv(Path('..') / 'datasets' / 'processed' / 'memory_features_train.csv')
X = train.drop(columns='target').to_numpy()
y = train['target'].to_numpy()

rng = np.random.default_rng(42)
W = rng.normal(0, 0.1, (X.shape[1], 4))   # (7 feature, 4 classi), pesi CASUALI

punteggi = X @ W                          # (213, 4)
predizioni = punteggi.argmax(axis=1)      # la classe col punteggio massimo
accuratezza = (predizioni == y).mean()

print(f'forme: X {X.shape} @ W {W.shape} -> punteggi {punteggi.shape}')
print(f'accuratezza con pesi casuali: {accuratezza:.0%}  (4 classi: il caso fa ~25%)')
print(f"quota della classe più frequente: {np.bincount(y).max() / len(y):.0%}")

Tieni a mente i due numeri stampati: il caso e la classe più frequente sono
le baseline oneste. Tutto ciò che il progetto imparerà d'ora in poi si
misura contro di loro.

## Cosa hai imparato

- Vettori, matrici e tensori sono lo stesso oggetto a dimensioni crescenti;
  la convenzione ML è `(esempi, feature)`.
- Il **prodotto scalare è una somma pesata** — un neurone è questo.
- `X @ W` produce tutti i punteggi di tutte le classi in un colpo:
  `(n, k) @ (k, m) -> (n, m)`, dimensioni interne uguali.
- Il broadcasting estende le forme compatibili (riga per riga).
- Prima domanda davanti a qualsiasi dato: *che forma ha?*

## Quiz

1. Un batch di 64 frasi, ognuna rappresentata da 300 numeri, entra in uno
   strato con matrice dei pesi (300, 128). Forme di input e output?
2. Perché `B @ A` fallisce se `A @ B` funziona?
3. Nel progetto, cosa rappresenta la colonna 2 della matrice `punteggi`?

<details>
<summary><b>Apri le risposte</b></summary>

1. Input `(64, 300)`, output `(64, 300) @ (300, 128) -> (64, 128)`: 128
   nuove feature per ciascuna delle 64 frasi.
2. La moltiplicazione richiede che le colonne del primo combacino con le
   righe del secondo: (3,5)@(4,3) ha 5 ≠ 4. Il prodotto di matrici non è
   commutativo.
3. Il punteggio della classe 2 (semantic) per ogni memoria: quanto il
   modello "crede" che ciascuna memoria sia semantic, prima di argmax.
</details>

## Fonti

- NumPy, *Broadcasting*:
  https://numpy.org/doc/stable/user/basics.broadcasting.html
- NumPy, `matmul` / operatore `@`:
  https://numpy.org/doc/stable/reference/generated/numpy.matmul.html

## Prossima lezione

I pesi casuali indovinano al 25%. Per fare meglio serve un modo di
**aggiustarli nella direzione giusta**: derivate, gradienti e la discesa che
muove tutto il deep learning. Lezione 8.